In [1]:
1+1

2

In [26]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [ ]:
from langsmith import Client 
client = Client()

dataset_name = "Chatbot Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id = dataset.id,
    examples = [
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['6319b7df-d760-4c63-aad1-85e2dd4f8472',
  'e0da9789-7113-4df2-9d75-7f2431b462ce',
  '42674516-b5b8-4ca3-b790-906374ae479f',
  '3d7a43e1-664f-4648-a70b-da3806b84b28',
  'a0a2fcce-7583-414e-ab5c-d92269b199b9'],
 'count': 5,
 'as_of': '2026-07-21T05:49:09.688462635Z'}

In [56]:
from google import genai
from google.genai import types
from langsmith import wrappers
import os

gemini_client = wrappers.wrap_gemini(
    genai.Client(api_key=os.environ["GEMINI_API_KEY"])
)

eval_instructions = (
    "You are an expert professor specialized in grading students' answers. "
    "Respond with ONLY one word: CORRECT or INCORRECT."
)

def correctness(inputs: dict, outputs: dict, reference_outputs: dict):
    user_content = f"""
Question:
{inputs['question']}

Reference Answer:
{reference_outputs['answer']}

Student Answer:
{outputs['response']}

Is the student answer correct?

Respond with ONLY:
CORRECT
or
INCORRECT
"""

    response = gemini_client.models.generate_content(
        model="gemini-3.5-flash",
        contents=user_content,
        config=types.GenerateContentConfig(
            system_instruction=eval_instructions,
            temperature=0,
        ),
    )

    grade = response.text.strip().upper()

    return {
        "key": "correctness",
        "score": 1 if grade == "CORRECT" else 0,
    }

In [57]:
def concision(inputs: dict, outputs: dict, reference_outputs: dict):
    return {
        "key": "concision",
        "score": int(
            len(outputs["response"]) < 2 * len(reference_outputs["answer"])
        ),
    }

In [58]:
from google.genai import types

default_instructions = (
    "Respond to the user's question in a short, concise manner (one short sentence)."
)

def my_app(
    question: str,
    model: str = "gemini-3.5-flash",
    instructions: str = default_instructions,
) -> str:

    response = gemini_client.models.generate_content(
        model=model,
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=instructions,
            temperature=0,
        ),
    )

    return response.text

In [59]:
#call my_app for every datapoints
def ls_target(inputs:str) -> dict:
    return {"response":my_app(inputs["question"])}

In [60]:
experiments_result = client.evaluate(
    ls_target,
    data = dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="genai-3.5-flash"
)

View the evaluation results for experiment: 'genai-3.5-flash-c7ad8654' at:
https://smith.langchain.com/o/45409f9a-a636-444b-828f-e8275b159680/datasets/aeff6aa6-ac20-4f91-83b6-38e737d89394/compare?selectedSessions=681c417b-c162-493b-aaf9-6b27ea66ea7b




3it [00:10,  3.61s/it]Error running target function: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.5-flash\nPlease retry in 6.735383063s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'model': '